In [ ]:
# 1. Verify GPU and Hugging Face login (same purpose as the sample notebook).
import torch
from google.colab import userdata
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
if torch.cuda.is_available():
    print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Hugging Face secret configured:", bool(HF_TOKEN))
except Exception:
    HF_TOKEN = None
    print("Hugging Face token not configured (private Hub verification will be skipped).")

In [ ]:
# 2. Download your GitHub project and install training/inference packages.
import subprocess, sys
from pathlib import Path
repo_dir = Path("/content/deepseek-finetuning")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Birajkakadiya/deepseek-finetuning.git",
                    str(repo_dir)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(repo_dir / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover",
                "-s", "tests", "-v"], cwd=str(repo_dir), check=True)
sys.path.insert(0, str(repo_dir))
print("GitHub code downloaded:", repo_dir)

In [ ]:
# 3. Preview the NVIDIA OpenCodeInstruct dataset (streaming).
from datasets import load_dataset
from data_utils import valid_record, record_to_conversation
dataset = load_dataset("nvidia/OpenCodeInstruct", split="train", streaming=True)
sample_row = next(row for row in dataset if valid_record(row))
formatted = record_to_conversation(sample_row)
print("Dataset: nvidia/OpenCodeInstruct")
print("Fields:", list(sample_row.keys()))
print("USER PROMPT:", formatted["prompt"][0]["content"][:400])
print("ASSISTANT ANSWER:", formatted["completion"][0]["content"][:400])
print("Only a streamed example was loaded; the full dataset is not downloaded.")

In [ ]:
# 4. Optional: perform a NEW 60-step LoRA run on a T4 (disabled by default).
# Your previously completed 2-step smoke test is already saved and does NOT need rerunning.
RUN_NEW_TRAINING = False
TRAIN_STEPS = 60
new_run_dir = repo_dir / "outputs" / "professor-t4-60steps"
if RUN_NEW_TRAINING:
    assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
    command = [sys.executable, str(repo_dir / "train.py"),
               "--preset", "t4", "--max-steps", str(TRAIN_STEPS),
               "--max-train-samples", "1500", "--max-eval-samples", "100",
               "--max-length", "1024", "--output-dir", str(new_run_dir)]
    subprocess.run(command, check=True, cwd=str(repo_dir))
else:
    print("No new training started. Using the verified 2-step saved adapter below.")

In [ ]:
# 5. Open your existing trained adapter and metrics from Google Drive.
import json
from zipfile import ZipFile
from google.colab import drive
drive.mount("/content/drive")
adapter_zip = Path("/content/drive/MyDrive/deepseek-coder-t4-smoke-adapter.zip")
assert adapter_zip.is_file(), "Move the saved adapter ZIP into MyDrive or update adapter_zip"
with ZipFile(adapter_zip) as z:
    assert z.testzip() is None, "Adapter archive ZIP is damaged"
    metadata = json.loads(z.read("run_metadata.json"))
    state = json.loads(z.read("trainer_state.json"))
    adapter_cfg = json.loads(z.read("adapter/adapter_config.json"))
print("BASE MODEL:", metadata["base_model"])
print("DATASET:", metadata["dataset"])
print("GPU:", metadata["gpu"])
print("METHOD: 4-bit QLoRA")
print("LoRA rank / alpha / dropout:", adapter_cfg["r"], adapter_cfg["lora_alpha"], adapter_cfg["lora_dropout"])
print("Training examples:", metadata["train_rows"], "Validation examples:", metadata["eval_rows"])
print("Training steps completed:", state["global_step"])
print("Training loss:", metadata["train_metrics"]["train_loss"])
print("Validation loss:", metadata["eval_metrics"]["eval_loss"])
print("Validation token accuracy:", metadata["eval_metrics"]["eval_mean_token_accuracy"])
print("NOTE: token accuracy is not the coding-problem pass rate.")

In [ ]:
# 6. Plot REAL saved training/validation loss values (from the 2-step smoke test).
import matplotlib.pyplot as plt
train_points = [(row["step"], row["loss"]) for row in state["log_history"] if "loss" in row]
val_points = [(row["step"], row["eval_loss"]) for row in state["log_history"] if "eval_loss" in row]
plt.figure(figsize=(8, 4.5))
if train_points:
    plt.plot(*zip(*train_points), marker="o", label="Training loss")
if val_points:
    plt.plot(*zip(*val_points), marker="s", label="Validation loss")
plt.title("DeepSeek-Coder 6.7B + NVIDIA OpenCodeInstruct — T4 QLoRA")
plt.xlabel("Optimizer steps")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("/content/deepseek-training-loss.png", dpi=160)
plt.show()
print("Graph saved as /content/deepseek-training-loss.png")

In [ ]:
# 7. Load the ORIGINAL DeepSeek-Coder base model and your trained LoRA adapter.
# This step requires a T4 GPU, but no new training.
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
adapter_dir = Path("/content/deepseek-saved-adapter/adapter")
adapter_dir.mkdir(parents=True, exist_ok=True)
with ZipFile(adapter_zip) as z:
    for name in z.namelist():
        if name.startswith("adapter/") and not name.endswith("/"):
            dest = adapter_dir / Path(name).name
            with z.open(name) as source, dest.open("wb") as target:
                shutil.copyfileobj(source, target)
tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir), use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-6.7b-instruct",
    quantization_config=quantization, device_map="auto",
    torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, str(adapter_dir)).eval()
print("Base model and LoRA adapter loaded on:", torch.cuda.get_device_name(0))

In [ ]:
# 8. Sample code-generation prompt: compare original vs fine-tuned model.
PROMPT = "Write a Python function for binary search, including 3 unit tests."
input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": PROMPT}],
    return_tensors="pt", add_generation_prompt=True).to("cuda")
def generate_answer(active_model):
    with torch.no_grad():
        output = active_model.generate(
            input_ids, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)
with model.disable_adapter():
    baseline_answer = generate_answer(model)
fine_tuned_answer = generate_answer(model)
print("===== ORIGINAL DEEPSEEK-CODER =====")
print(baseline_answer)
print("\n===== LORA ADAPTER =====")
print(fine_tuned_answer)
print("\nThis is a demonstration, not a benchmark.")

In [ ]:
# 9. Save a NEW training run (if you enabled Step 4) to Google Drive.
# Your existing smoke-test ZIP is already safely in MyDrive.
import shutil
if new_run_dir.exists() and (new_run_dir / "adapter").is_dir():
    destination = "/content/drive/MyDrive/DeepSeek_Coder_T4_60steps"
    shutil.copytree(str(new_run_dir), destination, dirs_exist_ok=True)
    print("New checkpoint saved to:", destination)
else:
    print("No new training checkpoint to save.")
print("Existing smoke-test ZIP:", adapter_zip)

In [ ]:
# 10. Submission links and private Hugging Face model verification.
GITHUB_URL = "https://github.com/Birajkakadiya/deepseek-finetuning"
COLAB_URL = "https://colab.research.google.com/github/Birajkakadiya/deepseek-finetuning/blob/main/colab/Professor_Submission_DeepSeek_Coder_6_7B.ipynb"
HF_ID = "Biraj999/deepseek-coder-6.7b-opencodeinstruct-lora-t4-smoke"
HF_URL = "https://huggingface.co/" + HF_ID
DRIVE_URL = "https://drive.google.com/file/d/19JGaUHzIDYAhkN-Vy7jycBqgHatu7eig/view"
print("GitHub:", GITHUB_URL)
print("Colab notebook:", COLAB_URL)
print("Hugging Face adapter (PRIVATE):", HF_URL)
print("Google Drive ZIP:", DRIVE_URL)
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    files = api.list_repo_files(repo_id=HF_ID, repo_type="model")
    print("Adapter uploaded:", "adapter_model.safetensors" in files)
    print("Model card uploaded:", "README.md" in files)
else:
    print("Private model verification skipped: no HF_TOKEN secret.")